In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import os
import glob
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

warnings.filterwarnings("ignore")

SEED = 42

In [ ]:
train_path = "/kaggle/input/competitions/survivor-detection-challenge-classification/maritime_train.csv"
test_path = "/kaggle/input/competitions/survivor-detection-challenge-classification/maritime_test.csv"

if not os.path.exists(train_path):
    files = glob.glob("/kaggle/input/**/*.csv", recursive=True)
    print("Available CSV files:")
    print("\n".join(files))
    raise FileNotFoundError("Update train_path using the printed file paths.")

train = pd.read_csv(train_path)

print("Training shape:", train.shape)
print(train.head())
print(train.columns.tolist())

In [ ]:
print("Missing values:")
print(train.isna().sum().sort_values(ascending=False))

print("\nTarget distribution:")
print(train["Outcome"].value_counts())
print(train["Outcome"].value_counts(normalize=True).round(3))

print("\nNumerical summary:")
print(train.describe().T.round(2))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.countplot(data=train, x="Outcome", ax=axes[0])
axes[0].set_title("Target distribution")

train.isna().sum().sort_values().plot(
    kind="barh", ax=axes[1]
)
axes[1].set_title("Missing values by column")

plt.tight_layout()
plt.show()

In [ ]:
def prepare_features(df):
    df = df.copy()

    if "PassengerName" in df.columns:
        names = df["PassengerName"].fillna("Unknown")

        df["Surname"] = names.str.split(",").str[0].str.strip()

        title = names.str.extract(r",\s*([^.]*)\.", expand=False)

        if "Title" in df.columns:
            df["Title"] = df["Title"].fillna(title)
        else:
            df["Title"] = title

    rounded_cols = [
        "TicketTier",
        "RelativesAboard",
        "ParentsChildren"
    ]

    for col in rounded_cols:
        if col in df.columns:
            df[col + "_rounded"] = df[col].round()

    if "Age" in df.columns:
        df["AgeGroup"] = pd.cut(
            df["Age"],
            bins=[-np.inf, 5, 12, 18, 30, 45, 60, np.inf],
            labels=[
                "Child", "YoungChild", "Teen",
                "YoungAdult", "Adult", "MiddleAge", "Senior"
            ]
        ).astype("object")

    if "FamilySize" in df.columns:
        df["FamilySizeGroup"] = pd.cut(
            df["FamilySize"],
            bins=[0, 1, 4, np.inf],
            labels=["Alone", "Small", "Large"]
        ).astype("object")

    for col in ["TicketCost", "FarePerPerson"]:
        if col in df.columns:
            df[col + "_log"] = np.log1p(df[col].clip(lower=0))

    df = df.drop(
        columns=["PassengerName", "PassengerId"],
        errors="ignore"
    )

    for col in df.select_dtypes(
        include=["object", "category", "bool"]
    ).columns:
        df[col] = df[col].fillna("Missing").astype(str)

    return df

In [ ]:
X = train.drop(columns=["Outcome"])
y = train["Outcome"].astype(int)

X = prepare_features(X)

constant_cols = [
    col for col in X.columns
    if X[col].nunique(dropna=False) <= 1
]

X = X.drop(columns=constant_cols)

cat_features = X.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()

print("Features:", X.shape[1])
print("Categorical features:", cat_features)
print("Target counts:")
print(y.value_counts())

In [ ]:
skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=SEED
)

oof_prob = np.zeros(len(X))
scores = []
models = []

for fold, (train_idx, valid_idx) in enumerate(skf.split(X, y), 1):
    X_train = X.iloc[train_idx]
    X_valid = X.iloc[valid_idx]
    y_train = y.iloc[train_idx]
    y_valid = y.iloc[valid_idx]

    model = CatBoostClassifier(
        iterations=1000,
        depth=5,
        learning_rate=0.03,
        loss_function="Logloss",
        eval_metric="Accuracy",
        l2_leaf_reg=5,
        random_seed=SEED,
        verbose=False,
        allow_writing_files=False
    )

    model.fit(
        X_train,
        y_train,
        cat_features=cat_features,
        eval_set=(X_valid, y_valid),
        early_stopping_rounds=100,
        verbose=False
    )

    prob = model.predict_proba(X_valid)[:, 1]
    pred = (prob >= 0.5).astype(int)

    oof_prob[valid_idx] = prob

    score = accuracy_score(y_valid, pred)
    scores.append(score)
    models.append(model)

    print(f"Fold {fold}: {score:.4f}")

print("\nMean fold accuracy:", round(np.mean(scores), 4))
print("Fold standard deviation:", round(np.std(scores), 4))
print("Overall OOF accuracy:", round(
    accuracy_score(y, (oof_prob >= 0.5).astype(int)), 4
))

print(classification_report(y, (oof_prob >= 0.5).astype(int)))

In [ ]:
oof_pred = (oof_prob >= 0.5).astype(int)

sns.heatmap(
    confusion_matrix(y, oof_pred),
    annot=True,
    fmt="d",
    cmap="Blues"
)

plt.title("Out-of-fold confusion matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

importance = pd.Series(
    models[-1].get_feature_importance(),
    index=X.columns
).sort_values(ascending=False)

print(importance.head(20))

importance.head(20).sort_values().plot(
    kind="barh",
    figsize=(9, 6)
)

plt.title("Feature importance")
plt.tight_layout()
plt.show()

In [ ]:
oof_pred = (oof_prob >= 0.5).astype(int)

sns.heatmap(
    confusion_matrix(y, oof_pred),
    annot=True,
    fmt="d",
    cmap="Blues"
)

plt.title("Out-of-fold confusion matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

importance = pd.Series(
    models[-1].get_feature_importance(),
    index=X.columns
).sort_values(ascending=False)

print(importance.head(20))

importance.head(20).sort_values().plot(
    kind="barh",
    figsize=(9, 6)
)

plt.title("Feature importance")
plt.tight_layout()
plt.show()

In [ ]:
test_files = glob.glob(
    "/kaggle/input/**/maritime_test.csv",
    recursive=True
)

if not test_files:
    raise FileNotFoundError(
        "Upload or attach maritime_test.csv to the Kaggle notebook."
    )

test = pd.read_csv(test_files[0])

if "PassengerName" not in test.columns:
    raise ValueError("PassengerName is missing from the test data.")

passenger_names = test["PassengerName"].copy()

X_test = prepare_features(test)
X_test = X_test.reindex(columns=X.columns)

for col in cat_features:
    X_test[col] = X_test[col].fillna("Missing").astype(str)

final_model = CatBoostClassifier(
    iterations=1000,
    depth=5,
    learning_rate=0.03,
    loss_function="Logloss",
    l2_leaf_reg=5,
    random_seed=SEED,
    verbose=False,
    allow_writing_files=False
)

final_model.fit(X, y, cat_features=cat_features)

test_prob = final_model.predict_proba(X_test)[:, 1]
test_pred = (test_prob >= 0.5).astype(int)

submission = pd.DataFrame({
    "PassengerName": passenger_names,
    "Outcome": test_pred
})

submission.to_csv("/kaggle/working/submission.csv", index=False)

print(submission.head())
print("Submission shape:", submission.shape)
print("Saved to /kaggle/working/submission.csv")